# Treinamento local e seleção

Notebook de análise de artefatos: não inicia downloads ou treinamento. Não substitui revisão humana de fundamentação.

## Execução real e reprodução
O treinamento foi executado pela CLI local e os registros abaixo são resultados dessa execução, não valores simulados. Implementação: `src/citizentech_ia/training.py`. Para repetir em uma nova rodada, com nome inédito e recursos livres, executar manualmente `citizentech train --name NOVO-NOME --lr 0.00001`. Não reexecutar o orquestrador sobre resultados existentes.

QLoRA sobre base 4 bits, rank 16 nas projeções q/v das últimas 16 camadas. Cada candidato usa 800 microsteps, acumulação 4 e 200 atualizações. A seleção de checkpoint usa validation loss; a seleção entre candidatos usa geração na validação, sem consultar o teste. Contextos supervisionados podem conter fonte de apoio do gabarito, com contagem declarada. Isso não ocorre na avaliação final. Rodadas v1 foram invalidadas por vazamento de identificador; somente v2 participa dos resultados.

In [1]:
import json
from pathlib import Path
for name in ['smoke-01','lr1e5','lr5e5']:
 p=Path('avaliacao/treinamento')/name/'summary.json'
 if not p.exists(): p=Path('adapters')/name/'summary.json'
 print(name, json.loads(p.read_text()) if p.exists() else 'PENDENTE')

smoke-01 {'microsteps': 40, 'optimizer_updates': 10, 'elapsed_s': 125.49134516716003, 'seconds_per_update': 12.549134612083435, 'peak_memory_gb': 8.381769864, 'adapters_changed': True, 'best_validation_loss': 1.3828413486480713, 'best_iteration': 40, 'final_validation_loss': 1.3828413486480713, 'adapter_sha': 'feab0f1d5e442f44375ec90a0e0d8deba4d928d552866425c9871ca10222aea5'}
lr1e5 {'microsteps': 800, 'optimizer_updates': 200, 'elapsed_s': 3149.518256187439, 'seconds_per_update': 15.747591285705566, 'peak_memory_gb': 8.565635568, 'adapters_changed': True, 'best_validation_loss': 0.07314658910036087, 'best_iteration': 800, 'final_validation_loss': 0.07314658910036087, 'adapter_sha': '066c7d9bc409e2adcd5cae1072ee05f03d73cd17f9008c4f5871ad178c99a38c'}
lr5e5 {'microsteps': 800, 'optimizer_updates': 200, 'elapsed_s': 3270.0286781787872, 'seconds_per_update': 16.350143395662307, 'peak_memory_gb': 8.56563572, 'adapters_changed': True, 'best_validation_loss': 0.018227407708764076, 'best_iterat

In [2]:
p=Path('configs/selected.json')
print(json.dumps(json.loads(p.read_text()),ensure_ascii=False,indent=2) if p.exists() else 'Seleção pendente.')

{
  "candidate": "lr1e5",
  "adapter": "adapters/selected/best",
  "criteria": "status_match, schema_and_refs, median_latency on validation only; technical proxy, not human semantic accuracy",
  "human_review": "pending",
  "validation_candidates": {
    "lr1e5": {
      "mode": "rag_finetuning",
      "split": "valid",
      "n": 80,
      "json_syntax_valid_rate": 1.0,
      "schema_valid_rate": 0.9375,
      "valid_json_and_refs_rate": 0.9375,
      "status_match_rate": 0.7125,
      "median_seconds": 2.509773083498658,
      "p95_seconds": 2.8747533124993425,
      "peak_memory_gb": 5.07032084,
      "semantic_correctness": "PENDENTE: status_match não é acerto semântico ou fundamentação.",
      "model": {
        "modelo": "Qwen/Qwen2.5-7B-Instruct",
        "revisao_modelo": "a09a35458c702b33eeacc393d103063234e8bc28",
        "adaptador": "066c7d9bc409e2adcd5cae1072ee05f03d73cd17f9008c4f5871ad178c99a38c"
      },
      "base_sha": "293f559b96abd075ae50158d866a94e05f6a882a51cb5910

In [3]:
for name in ['lr1e5','lr5e5']:
 p=Path('avaliacao/treinamento')/name/'provenance.json'
 if not p.exists(): p=Path('adapters')/name/'provenance.json'
 if p.exists():
  provenance=json.loads(p.read_text())
  print(name, 'prompt:',provenance.get('prompt_version'), 'versões:',provenance['versions'])
  print(json.dumps(provenance['settings'],ensure_ascii=False,indent=2))
print('Conversas supervisionadas:',json.loads(Path('dados/mlx/manifest.json').read_text()))

lr1e5 prompt: v2-no-identifiers versões: {'mlx': '0.32.4', 'mlx-lm': '0.32.2', 'transformers': '5.22.2', 'numpy': '2.5.4'}
{
  "model": "/Users/lucasmanentti/Projects/citizentech-iluminacao-ia/models/qwen-7b-4bit",
  "train": true,
  "fine_tune_type": "lora",
  "optimizer": "adamw",
  "optimizer_config": {
    "adamw": {
      "weight_decay": 0.01
    }
  },
  "data": "/Users/lucasmanentti/Projects/citizentech-iluminacao-ia/dados/mlx",
  "seed": 42,
  "num_layers": 16,
  "batch_size": 1,
  "iters": 800,
  "val_batches": -1,
  "learning_rate": 1e-05,
  "steps_per_report": 20,
  "steps_per_eval": 100,
  "resume_adapter_file": null,
  "adapter_path": "/Users/lucasmanentti/Projects/citizentech-iluminacao-ia/adapters/lr1e5",
  "save_every": 100,
  "test": false,
  "test_batches": 500,
  "max_seq_length": 2048,
  "config": null,
  "grad_checkpoint": false,
  "grad_accumulation_steps": 4,
  "clear_cache_threshold": 0,
  "lr_schedule": null,
  "lora_parameters": {
    "rank": 16,
    "scale": 